# Entraînement YOLOv8n 640 px sur GPU (Google Colab)

Projet **Détection de déchets pour le tri sélectif** · Tâche 4 · Syrine

Ce notebook récupère le code (GitHub) et les données (DagsHub), entraîne YOLOv8n en 640 px sur le GPU de Colab,
et envoie le run dans **MLflow sur DagsHub**. À la fin, il télécharge le fichier de résultats à commiter.

**Avant de lancer :**
1. Menu **Exécution → Modifier le type d'exécution → GPU T4** → Enregistrer.
2. Icône **clé** (Secrets) dans la barre de gauche → ajouter, avec l'accès au notebook activé :
   - `DAGSHUB_USER` : ton nom d'utilisateur DagsHub
   - `DAGSHUB_TOKEN` : ton token DagsHub
   - `GITHUB_TOKEN` : seulement si le dépôt GitHub est **privé**
3. Lancer les cellules **une par une**, de haut en bas.

Les secrets ne sont jamais écrits dans le notebook : ne colle jamais un token dans une cellule.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi

## 2. Réglages (à adapter si besoin)

In [ ]:
GITHUB_REPO = "Sirinebbj123/waste-sorting-mlops"   # propriétaire/nom du dépôt GitHub
DAGSHUB_REPO = "Sirinebbj123504/waste-sorting-mlops"  # propriétaire/nom du dépôt DagsHub
BRANCHE = "exp/yolo-640-colab"                      # la branche qui contient train_yolo_640.yaml
CONFIG = "ml/configs/train_yolo_640.yaml"

## 3. Lire les secrets et récupérer le code

In [ ]:
import os
from google.colab import userdata

DAGSHUB_USER = userdata.get("DAGSHUB_USER")
DAGSHUB_TOKEN = userdata.get("DAGSHUB_TOKEN")
try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = None  # dépôt public : pas besoin de token

url = f"https://github.com/{GITHUB_REPO}.git"
if GITHUB_TOKEN:
    url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"

%cd /content
!rm -rf waste-sorting-mlops
!git clone -q -b {BRANCHE} {url} waste-sorting-mlops
%cd /content/waste-sorting-mlops
!git log --oneline -1

## 4. Installer les outils (2 à 3 minutes)

In [ ]:
!pip install -q ultralytics mlflow dvc pandas pyyaml

## 5. Récupérer les données depuis DagsHub (DVC)

`dvc pull prepare_roboflow` télécharge le dossier `ml/data/processed/roboflow` exactement dans la version
indiquée par `dvc.lock`.

In [ ]:
!dvc remote modify origin --local auth basic
!dvc remote modify origin --local user {DAGSHUB_USER}
!dvc remote modify origin --local password {DAGSHUB_TOKEN}
!dvc pull prepare_roboflow
!ls ml/data/processed/roboflow

## 6. Entraîner (30 époques, environ 20 à 40 minutes sur T4)

Les variables `MLFLOW_*` envoient le run dans l'onglet **Experiments** de DagsHub.
Si la session Colab se coupe, relance depuis la cellule 3.

In [ ]:
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{DAGSHUB_REPO}.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN

!python ml/src/train/train_yolo.py --config {CONFIG}

## 7. Télécharger les résultats

- `yolo_yolov8n_640.json` : à copier dans `ml/reports/` sur ton PC, puis à commiter.
- `best.pt` : les poids (hors Git, déjà enregistrés dans le run MLflow sur DagsHub). Garde-le pour la démo.

In [ ]:
from google.colab import files
files.download("ml/reports/yolo_yolov8n_640.json")
files.download("ml/runs/yolov8n_640/weights/best.pt")